# Breachpoint v2: Granite analyst batch on vLLM (Google Colab)

Runs **IBM Granite 4.2-3B** behind **vLLM's OpenAI-compatible server** over ~137k vulnerability descriptions and saves structured fields (product category, internet-facing, login needed, attack type...).

**Before you start**
1. *Runtime → Change runtime type → **T4 GPU***
2. Copy `granite_job.zip` (from `outputs/v2_Granite_Analyst/` in the repo) to `MyDrive/breachpoint_granite/` on Google Drive. It holds `queue.parquet`, `prompt.py`, `extract.py`.
3. Results go to Google Drive (`MyDrive/breachpoint_granite/`), so a disconnect loses at most one chunk. **Re-run all cells to resume**; finished rows are skipped.

Expected: a few minutes to start, then roughly 1.5-3 hours for the full queue on a T4. Rows run in priority order, so a partial run still covers the most likely vulnerabilities in every cohort.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/breachpoint_granite'
zipfile.ZipFile(f'{WORK}/granite_job.zip').extractall(WORK)
%cd {WORK}
!ls -la
!grep -n "PROMPT_VERSION =" prompt.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/breachpoint_granite
total 61862
drwx------  6 root root     4096 Nov  8 08:47 .
drwx------ 20 root root     4096 Nov  8 07:05 ..
-rw-------  1 root root     5432 Nov  8 08:47 extract.py
drwx------  2 root root     4096 Nov  8 08:40 granite_fields
-rw-------  1 root root 29298745 Nov  8 08:04 granite_job.zip
drwx------  2 root root     4096 Nov  8 07:41 .ipynb_checkpoints
-rw-------  1 root root    19349 Nov  8 08:47 pilot_queue.parquet
-rw-------  1 root root    27721 Nov  8 08:40 pilot_results.csv
drwx------  2 root root     4096 Nov  8 08:40 pilot_run
-rw-------  1 root root     9469 Nov  8 08:47 prompt.py
drwx------  2 root root     4096 Nov  8 08:40 __pycache__
-rw-------  1 root root 33959558 Nov  8 08:47 queue.parquet
13:PROMPT_VERSION = "v3"


In [3]:
!pip install -q "vllm>=0.20" openai pandas pyarrow

## Start the vLLM server
T4 GPUs don't support bfloat16, so the model runs in float16 (`--dtype half`). Prefix caching reuses the long, shared system prompt across requests.

In [4]:
!sed -n '26,60p' /content/vllm.log | cut -c1-170 | grep -v shutdown | tail -15
!cp /content/vllm.log /content/vllm_first.log; pkill -f "vllm serve"; sleep 4; free -h | head -2
!setsid nohup vllm serve ibm-granite/granite-4.2-3b --served-model-name granite-4.2-3b --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --enable-prefix-caching --max-num-seqs 64 > /content/vllm.log 2>&1 < /dev/null &

(EngineCore pid=33536) WARNING 11-08 08:45:57 [utils.py:333] Attempted to load weight layers.11.post_attention_layernorm.weight with dtype torch.bfloat16 into parameter w
(EngineCore pid=33536) WARNING 11-08 08:45:57 [utils.py:333] Attempted to load weight layers.11.self_attn.o_proj.weight with dtype torch.bfloat16 into parameter with dtyp
(EngineCore pid=33536) WARNING 11-08 08:45:57 [utils.py:333] Attempted to load weight layers.12.input_layernorm.weight with dtype torch.bfloat16 into parameter with dtype
(EngineCore pid=33536) WARNING 11-08 08:45:57 [utils.py:333] Attempted to load weight layers.12.mlp.down_proj.weight with dtype torch.bfloat16 into parameter with dtype t
(EngineCore pid=33536) WARNING 11-08 08:45:58 [utils.py:333] Attempted to load weight layers.12.post_attention_layernorm.weight with dtype torch.bfloat16 into parameter w
(EngineCore pid=33536) WARNING 11-08 08:45:58 [utils.py:333] Attempted to load weight layers.12.self_attn.o_proj.weight with dtype torch.bfloat16

In [5]:
import time, urllib.request
for i in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2); print('vLLM ready'); break
    except Exception:
        time.sleep(10)
else:
    print('Not ready after 20 min; see the log below')
!tail -n 20 /content/vllm.log

vLLM ready
(APIServer pid=34331) INFO 11-08 08:49:19 [launcher.py:197] Route: /v1/messages/count_tokens, Methods: POST
(APIServer pid=34331) INFO 11-08 08:49:19 [launcher.py:197] Route: /generative_scoring, Methods: POST
(APIServer pid=34331) INFO 11-08 08:49:19 [launcher.py:197] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=34331) INFO 11-08 08:49:19 [launcher.py:197] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=34331) INFO 11-08 08:49:19 [launcher.py:197] Route: /v1/chat/completions/render, Methods: POST
(APIServer pid=34331) INFO 11-08 08:49:19 [launcher.py:197] Route: /v1/messages/render, Methods: POST
(APIServer pid=34331) INFO 11-08 08:49:19 [launcher.py:197] Route: /v1/completions/render, Methods: POST
(APIServer pid=34331) INFO 11-08 08:49:19 [launcher.py:197] Route: /v1/responses/render, Methods: POST
(APIServer pid=34331) INFO 11-08 08:49:19 [launcher.py:197] Route: /v1/chat/completions/derender, Methods: POST
(APIServer pid=34331) INFO 11-08 08:49:19 

## Pilot (60 rows): check the prompt before the full run
20 known later-exploited vulns + 40 random ones from 2024-25. Check that `ok` is ~100% (valid JSON) and the fields look right.
`pilot_results.csv` is saved to the Drive folder for checking by hand.
If outputs look garbled in float16, restart the server with `--dtype float32` (slower).

In [6]:
!python extract.py --queue pilot_queue.parquet --out pilot_run --concurrency 32 --chunk 60
import pandas as pd, glob
t = pd.concat(pd.read_parquet(p) for p in glob.glob('pilot_run/part-*.parquet'))
t = t.merge(pd.read_parquet('pilot_queue.parquet')[['cve_id', 'description']], on='cve_id')
print('valid JSON:', t.ok.mean())
t.to_csv('pilot_results.csv', index=False)
print('Saved to Drive:', WORK + '/pilot_results.csv')
pd.set_option('display.max_colwidth', 120)
t[['cve_id', 'category', 'internet_facing', 'pre_auth', 'attack_type', 'server_product', 'product', 'description']].head(20)

0 already done; 60 to go
60/60 done, 100.0% valid, 1.6/s, ~0 min left
valid JSON: 0.5
Saved to Drive: /content/drive/MyDrive/breachpoint_granite/pilot_results.csv


,cve_id,category,internet_facing,pre_auth,attack_type,server_product,product,description
0,CVE-2024-5942,None,None,None,None,None,None,The Page and Post Clone plugin for WordPress is vulnerable to Insecure Direct Object Reference in all versions up to...
1,CVE-2024-35671,None,None,None,None,None,None,Missing Authorization vulnerability in Minoji MJ Update History.This issue affects MJ Update History: from n/a throu...
2,CVE-2025-39784,None,None,None,None,None,None,"In the Linux kernel, the following vulnerability has been resolved: PCI: Fix link speed calculation on retrain failu..."
3,CVE-2025-20633,None,None,None,None,None,None,"In wlan AP driver, there is a possible out of bounds write due to an incorrect bounds check. This could lead to remo..."
4,CVE-2025-32433,None,None,None,None,None,None,"Erlang/OTP is a set of libraries for the Erlang programming language. Prior to versions OTP-27.3.3, OTP-26.2.5.11, a..."
5,CVE-2024-8957,None,None,None,None,None,None,PTZOptics PT30X-SDI/NDI-xx before firmware 6.3.40 is vulnerable to an OS command injection issue. The camera does no...
6,CVE-2025-60139,None,None,None,None,None,None,Cross-Site Request Forgery (CSRF) vulnerability in Joovii Sendle Shipping official-sendle-shipping-method allows Cro...
7,CVE-2025-39732,None,None,None,None,None,None,"In the Linux kernel, the following vulnerability has been resolved: wifi: ath11k: fix sleeping-in-atomic in ath11k_m..."
8,CVE-2024-7832,None,None,None,None,None,None,"** UNSUPPORTED WHEN ASSIGNED ** A vulnerability was found in D-Link DNS-120, DNR-202L, DNS-315L, DNS-320, DNS-320L, ..."
9,CVE-2024-2382,None,None,None,None,None,None,The Authorize.net Payment Gateway For WooCommerce plugin for WordPress is vulnerable to payment bypass in all versio...


## Full run (resumable)

In [ ]:
!python extract.py --queue queue.parquet --out granite_fields --concurrency 128 --chunk 2000

0 already done; 162,985 to go
2,000/162,985 done, 100.0% valid, 2.3/s, ~1165 min left
4,000/162,985 done, 100.0% valid, 2.3/s, ~1167 min left
6,000/162,985 done, 100.0% valid, 2.3/s, ~1140 min left
8,000/162,985 done, 100.0% valid, 2.3/s, ~1109 min left
10,000/162,985 done, 100.0% valid, 2.4/s, ~1047 min left
12,000/162,985 done, 100.0% valid, 2.5/s, ~1024 min left
14,000/162,985 done, 100.0% valid, 2.5/s, ~1007 min left
16,000/162,985 done, 100.0% valid, 2.5/s, ~979 min left
18,000/162,985 done, 100.0% valid, 2.5/s, ~964 min left
20,000/162,985 done, 100.0% valid, 2.5/s, ~959 min left
22,000/162,985 done, 100.0% valid, 2.5/s, ~949 min left
24,000/162,985 done, 100.0% valid, 2.5/s, ~935 min left
26,000/162,985 done, 100.0% valid, 2.5/s, ~921 min left
28,000/162,985 done, 100.0% valid, 2.5/s, ~908 min left
30,000/162,985 done, 100.0% valid, 2.5/s, ~893 min left
32,000/162,985 done, 100.0% valid, 2.5/s, ~881 min left


## Package the results
Get `granite_fields.zip` from the Drive folder and unzip it into `outputs/v2_Granite_Analyst/` in the repo (or copy the folder from Drive).

In [ ]:
!cd {WORK} && zip -qr granite_fields.zip granite_fields && ls -la granite_fields.zip
import pandas as pd, glob
r = pd.concat(pd.read_parquet(p) for p in glob.glob(f'{WORK}/granite_fields/part-*.parquet'))
print(len(r), 'rows,', round(r.ok.mean()*100, 1), '% valid'); print(r.category.value_counts().head(10))
print('Saved to Drive:', WORK + '/granite_fields.zip')